In [3]:
import pandas as pd
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, Image
)
from reportlab.lib import colors
from reportlab.lib import styles
from reportlab.lib.units import inch
from reportlab.platypus import TableStyle
from reportlab.pdfgen import canvas

# -------------------------
# CORES DA MARCA
# -------------------------
cor_primaria = colors.HexColor("#0A2E6E")   # Azul escuro
cor_secundaria = colors.HexColor("#66B2E8") # Azul claro

# -------------------------
# Função Rodapé
# -------------------------
def adicionar_rodape(canvas, doc):
    canvas.saveState()
    canvas.setFont('Helvetica', 8)
    canvas.setFillColor(colors.grey)
    
    canvas.drawString(40, 20, "www.carreiraemdados.com")
    canvas.drawRightString(550, 20, "@data_bruno")
    
    canvas.restoreState()

# -------------------------
# Importar planilhas
# -------------------------
vendas = pd.read_csv("vendas.csv")
metas = pd.read_csv("metas.csv")

# -------------------------
# Calcular KPIs
# -------------------------
resumo = vendas.groupby("vendedor").agg({
    "valor_venda": "sum",
    "comissao_individual": "sum",
    "id_venda": "count"
}).reset_index()

resumo.rename(columns={
    "valor_venda": "total_vendido",
    "comissao_individual": "comissao_total",
    "id_venda": "qtd_vendas"
}, inplace=True)

resumo = resumo.merge(metas, on="vendedor")
resumo["percentual_meta"] = resumo["total_vendido"] / resumo["meta_mensal"]

# -------------------------
# Gerar PDF por vendedor
# -------------------------
for _, row in resumo.iterrows():

    vendedor = row["vendedor"]

    doc = SimpleDocTemplate(
        f"Relatorio_{vendedor}.pdf",
        rightMargin=40,
        leftMargin=40,
        topMargin=40,
        bottomMargin=60
    )

    elements = []
    estilo = styles.getSampleStyleSheet()

    # -------------------------
    # LOGO
    # -------------------------
    logo = Image(r"C:\Users\Bruno Melo\Downloads\carreira_em_dados.png", width=2.5 * inch, height=1 * inch)
    elements.append(logo)
    elements.append(Spacer(1, 0.3 * inch))

    # -------------------------
    # TÍTULO
    # -------------------------
    elements.append(
        Paragraph(
            f"<font size=16 color='{cor_primaria}'>Relatório de Comissão</font>",
            estilo["Title"]
        )
    )
    elements.append(
        Paragraph(
            f"<b>Vendedor:</b> {vendedor}",
            estilo["Normal"]
        )
    )
    elements.append(Spacer(1, 0.4 * inch))

    # -------------------------
    # KPIs
    # -------------------------
    kpis = [
        ["Indicador", "Valor"],
        ["Total Vendido", f"R$ {row['total_vendido']:,.2f}"],
        ["Meta Mensal", f"R$ {row['meta_mensal']:,.2f}"],
        ["% da Meta", f"{row['percentual_meta']:.2%}"],
        ["Quantidade de Vendas", int(row["qtd_vendas"])],
        ["Comissão Total", f"R$ {row['comissao_total']:,.2f}"]
    ]

    tabela_kpi = Table(kpis, colWidths=[3 * inch, 2 * inch])

    tabela_kpi.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), cor_primaria),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
        ('FONTNAME', (0, 0), (-1, -1), 'Helvetica'),
        ('FONTSIZE', (0, 0), (-1, -1), 10)
    ]))

    elements.append(tabela_kpi)
    elements.append(Spacer(1, 0.5 * inch))

    # -------------------------
    # TABELA DETALHADA
    # -------------------------
    vendas_vendedor = vendas[vendas["vendedor"] == vendedor]

    dados_tabela = [["ID Venda", "Item", "Cliente", "Valor Venda", "Comissão"]]

    for _, venda in vendas_vendedor.iterrows():
        dados_tabela.append([
            venda["id_venda"],
            venda["item_vendido"],
            venda["cliente"],
            f"R$ {venda['valor_venda']:,.2f}",
            f"R$ {venda['comissao_individual']:,.2f}"
        ])

    tabela_detalhe = Table(dados_tabela, repeatRows=1)

    tabela_detalhe.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), cor_secundaria),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('GRID', (0, 0), (-1, -1), 0.3, colors.grey),
        ('FONTSIZE', (0, 0), (-1, -1), 8),
        ('ALIGN', (3, 1), (-1, -1), 'RIGHT')
    ]))

    elements.append(tabela_detalhe)

    # -------------------------
    # Gerar PDF
    # -------------------------
    doc.build(
        elements,
        onFirstPage=adicionar_rodape,
        onLaterPages=adicionar_rodape
    )

print("Relatórios gerados com sucesso!")

Relatórios gerados com sucesso!
